# Modeling

Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [85]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score

import joblib
from tqdm import tqdm
from sklearn.model_selection import TimeSeriesSplit

Load WTI Data

In [2]:
wti = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\wti_v3.parquet")

Load Tweet Data

In [6]:
tweets = pd.read_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\00_data\tweets_v6.parquet")

In [7]:
tweets.shape

(15242, 53)

In [8]:
tweets.columns

Index(['timestamp_utc', 'clean_text', 'embedding', 'finbert_positive',
       'finbert_neutral', 'finbert_negative', 'finbert_sentiment_score',
       'finbert_entropy', 'finbert_confidence', 'financial_uncertainty_score',
       'financial_risk_sentiment', 'finbert_polarization', 'caps_ratio',
       'exclamation_count', 'exclamation_count_log', 'tweet_length',
       'tweet_length_log', 'market_aggression_index',
       'time_since_last_tweet_min', 'rolling_tweet_frequency_60m',
       'rolling_tweet_frequency_6h', 'rolling_tweet_frequency_24h',
       'tweet_burst_indicator', 'tweet_acceleration_6h',
       'sentiment_delta_vs_previous', 'rolling_sentiment_mean_60m',
       'rolling_sentiment_std_60m', 'is_trump_president', 'wti_bullish_score',
       'wti_bearish_score', 'energy_supply_score',
       'geopolitical_oil_risk_score', 'usd_strength_pressure_score',
       'fed_monetary_policy_score', 'risk_sentiment_score',
       'trump_energy_policy_score', 'trump_market_shock_langua

In [9]:
wti.shape

(2601773, 76)

In [11]:
wti.columns

Index(['timestamp_utc', 'open', 'high', 'low', 'close', 'log_close', 'ret_1',
       'hl_range', 'oc_return', 'ret_5', 'ret_15', 'ret_30', 'ret_60',
       'ret_240', 'mom_5', 'mom_15', 'mom_60', 'vol_close_5',
       'vol_parkinson_5', 'vol_close_15', 'vol_parkinson_15', 'vol_close_30',
       'vol_parkinson_30', 'vol_close_60', 'vol_parkinson_60', 'vol_close_240',
       'vol_parkinson_240', 'vol_gk', 'vol_ratio_5_60', 'sma_5', 'ema_5',
       'dist_ema_5', 'sma_15', 'ema_15', 'dist_ema_15', 'sma_60', 'ema_60',
       'dist_ema_60', 'sma_240', 'ema_240', 'dist_ema_240', 'hl_momentum',
       'body', 'upper_wick', 'lower_wick', 'abs_ret', 'vol_cluster', 'vol_5',
       'jump', 'hour', 'dayofweek', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'us_session', 'vol_rank', 'vol_regime', 'trend_60',
       'trend_regime', 'y_up_2', 'y_down_2', 'y_up_5', 'y_down_5', 'y_up_15',
       'y_down_15', 'y_up_30', 'y_down_30', 'y_up_60', 'y_down_60', 'y_up_240',
       'y_down_240', 'y_up_72

## Train-Validation-Test-Split

In [20]:
tweets_pre2020 = tweets[tweets['timestamp_utc'] < '2020-01-01'] # wird für training und validation verwendet
tweets_post2020 = tweets[tweets['timestamp_utc'] >= '2020-01-01'] # als test set für die out of sample performance

## ML-Pipeline

### 1. Config

In [34]:
PCA_COMPONENTS = 20
N_SPLITS = 5
PURGE_MINUTES = 1440

TIME_WTI = "timestamp_utc"
TIME_TWEET = "timestamp_utc"

### 2. Embedding Parser

In [35]:
def parse_embedding(x):
    return np.fromstring(x.strip("[]"), sep=" ")

### 3. Tweet Preprocessing + Embeddings

In [36]:
def prepare_tweets(df):

    df = df.copy()

    df[TIME_TWEET] = pd.to_datetime(df[TIME_TWEET], utc=True)

    df["embedding_vec"] = df["embedding"].apply(parse_embedding)

    emb = np.vstack(df["embedding_vec"].values)

    emb_cols = [f"emb_{i}" for i in range(emb.shape[1])]
    emb_df = pd.DataFrame(emb, columns=emb_cols)

    df = pd.concat([df.reset_index(drop=True), emb_df], axis=1)

    return df

### 4. Tweet Aggregation

Notiz: Hier fehlen noch die aggregation für die restlichen spalten! also unvollständig

In [37]:
def aggregate_tweets(df):

    df = df.copy()

    df["minute"] = df[TIME_TWEET].dt.floor("min")

    emb_cols = [c for c in df.columns if c.startswith("emb_")]

    agg_dict = {
        "finbert_sentiment_score": ["mean", "std"],
        "financial_uncertainty_score": ["mean", "max"],
        "tweet_length": ["mean", "sum"],
        "exclamation_count": ["mean"]
    }

    for c in emb_cols:
        agg_dict[c] = ["mean", "max", "min"]

    agg = df.groupby("minute").agg(agg_dict)

    agg.columns = ["_".join(col) for col in agg.columns]
    agg = agg.reset_index()

    return agg

### 5. Leakage-Free WTI Mapping

In [39]:
def map_to_wti(tweet_agg, wti):

    tweet_agg = tweet_agg.copy()
    wti = wti.copy()

    tweet_agg[TIME_TWEET] = pd.to_datetime(tweet_agg["minute"], utc=True)
    wti[TIME_WTI] = pd.to_datetime(wti[TIME_WTI], utc=True)

    # KEY RULE:
    # tweet -> previous completed minute candle
    tweet_agg["wti_time"] = tweet_agg["minute"] - pd.Timedelta(minutes=1)

    df = tweet_agg.merge(
        wti,
        left_on="wti_time",
        right_on=TIME_WTI,
        how="left"
    )

    df = df.dropna(subset=["open"])

    return df

### 6. Feature / Target Split

In [41]:
TARGETS = [c for c in wti.columns if c.startswith("y_")]

def split_xy(df):

    y = df[TARGETS]
    X = df.drop(columns=TARGETS)

    return X, y

### 7. Purged CV

## nutze lieber time series split - ist eine getestete bibliothek

In [ ]:
# class PurgedCV:

#     def __init__(self, n_splits=5, purge=1440):
#         self.n_splits = n_splits
#         self.purge = purge

#     def split(self, X):

#         n = len(X)
#         idx = np.arange(n)
#         fold = n // self.n_splits

#         for i in range(self.n_splits):

#             val_start = i * fold
#             val_end = n if i == self.n_splits - 1 else (i + 1) * fold

#             val_idx = idx[val_start:val_end]

#             train_mask = np.ones(n, dtype=bool)

#             train_mask[
#                 max(0, val_start - self.purge):
#                 min(n, val_end + self.purge)
#             ] = False

#             train_idx = idx[train_mask]

#             yield train_idx, val_idx

### 8. Models

In [43]:
MODELS = {
    "logreg": LogisticRegression(
        penalty="l1",
        solver="saga",
        max_iter=5000,
        class_weight="balanced"
    ),

    "rf": RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        random_state=42
    ),

    "mlp": MLPClassifier(
        hidden_layer_sizes=(128, 64),
        max_iter=300
    )
}

### 9. Evaluation

In [44]:
def evaluate(y_true, pred, prob):

    return {
        "roc_auc": roc_auc_score(y_true, prob),
        "f1": f1_score(y_true, pred),
        "acc": accuracy_score(y_true, pred)
    }

In [ ]:
def clean_features(df):

    # print(df.isna().sum()[df.isna().sum() > 0])

    # nur numerische Features
    df = df.select_dtypes(include=[np.number])

    # df = df.dropna()

    return df


### 10. Core Training Loop

To Dos:
- aggregate tweets macht das noch nicht mit allen features, muss ich nachbessern
- bei feature_sets werden aktuell nur die embeddings gedroppt und baseline enthält damit noch tweet features
- wird nicht pca transfomiert stand jetzt
- feature selection
- feature importance + diagnostik fehlt
- hyperparameter tuning fehlt
- modelle müssen angepasst werden - MLP bewusster konfigurieren etc., early stopping etc. pp
- overfitting detecten und beheben
- andere metriken nehmen
- in dataframe wegschreiben und im nachgang analysierenm, auch feature importance, residen, testing etc.
- Visualisierungen bei timerseries split, learning curves etc.


In [ ]:
def run_pipeline(wti_df, tweets_df):

    results = []
    models_store = {}

    tweets_df = prepare_tweets(tweets_df)
    tweets_agg = aggregate_tweets(tweets_df)

    df = map_to_wti(tweets_agg, wti_df)
    df = df.sort_values("minute").reset_index(drop=True)

    X, y = split_xy(df)
    
    # Beginn von mir ergänzer Code
    std_cols = [c for c in X.columns if c.endswith("_std")]
    # print(std_cols)
    for c in std_cols:
        X[c] = X[c].fillna(0)
    # Ende von mir ergänzer Code

    # feature_sets = {
    #     "baseline": X.drop(columns=[c for c in X.columns if c.startswith("emb_")]),
    #     "enhanced": X
    # }
    feature_sets = {
        "baseline": clean_features(
            X.drop(columns=[c for c in X.columns if c.startswith("emb_")])
        ),
        "enhanced": clean_features(X)
    }
    # cv = PurgedCV(n_splits=N_SPLITS, purge=PURGE_MINUTES)
    cv = TimeSeriesSplit(
        n_splits=N_SPLITS,
        gap=PURGE_MINUTES
    )

    for target in TARGETS:

        y_t = y[target]

        for fs_name, X_fs in feature_sets.items():

            for model_name, model in MODELS.items():

                for fold, (tr, va) in enumerate(cv.split(X_fs)):

                    X_train, X_val = X_fs.iloc[tr], X_fs.iloc[va]
                    y_train, y_val = y_t.iloc[tr], y_t.iloc[va]

                    # print(X_train.columns)
                    # print(X_train.dtypes)
                    # X_train.drop("minute", axis=1, inplace=True)
                    # X_val.drop("minute", axis = 1, inplace = True)

                    scaler = StandardScaler()
                    X_train = scaler.fit_transform(X_train)
                    X_val = scaler.transform(X_val)

                    model.fit(X_train, y_train)

                    prob = model.predict_proba(X_val)[:, 1]
                    pred = (prob > 0.5).astype(int)

                    scores = evaluate(y_val, pred, prob)

                    results.append({
                        "target": target,
                        "feature_set": fs_name,
                        "model": model_name,
                        "fold": fold,
                        **scores
                    })

                    models_store[(target, fs_name, model_name, fold)] = model

    return pd.DataFrame(results), models_store

hier fehlt noch die training scores auszuwerten, nicht nur die validations damit man overfitting etc. erkennen kann

### 11. Best Model Export

In [56]:
def save_best(results, models_store):

    best = results.sort_values("roc_auc", ascending=False).iloc[0]

    key = (
        best.target,
        best.feature_set,
        best.model,
        best.fold
    )

    joblib.dump(models_store[key], "best_model.pkl")

    return best

In [74]:
std_cols = [c for c in tweets_pre2020.columns if c.endswith("_std")]
print(std_cols)

[]


### 12. RUN

In [77]:
results, models = run_pipeline(wti, tweets_pre2020)

best_model_info = save_best(results, models)

print(best_model_info)

C:\Users\02_Florian_Benutzer\AppData\Local\Temp\ipykernel_17404\1248267005.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  agg = agg.reset_index()


Series([], dtype: int64)
Series([], dtype: int64)


c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use 

target         y_down_2
feature_set    baseline
model                rf
fold                  4
roc_auc        0.600463
f1             0.095745
acc            0.702016
Name: 39, dtype: object


In [84]:
results.to_parquet(r"C:\Users\02_Florian_Benutzer\Desktop\trump_oil_analysis_v2\results\results_v1.parquet", index=False)

In [89]:
len(models)

480